In [1]:
#install + imports
!pip install -q -U bitsandbytes peft accelerate transformers
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import json, re, time
import torch
from pathlib import Path
from google.colab import drive, files, userdata
from huggingface_hub import login

drive.mount('/content/drive')
login(token=userdata.get('HF_TOKEN'))

#CHECKPOINT_DIR = Path("/content/drive/MyDrive/finetuning_checkpoints")
CHECKPOINT_DIR = Path("/content/drive/MyDrive/finetuning_checkpoints_qwen")#change the paths for qwen
ADAPTER_PATH = CHECKPOINT_DIR / "final_adapter"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 27.7 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# upload eval_context.json from the local notebook
uploaded = files.upload()
with open("eval_context.json", encoding="utf-8") as f:
    eval_context = json.load(f)
print(f"{len(eval_context)} eval queries with context loaded")

Saving eval_context.json to eval_context.json
15 eval queries with context loaded


In [3]:
#load base model (comparison baseline)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

#MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="auto",
)
print("Base model loaded")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Base model loaded


In [4]:
# generation helper, shared by both models
CITATION_SYSTEM_PROMPT = """You are a question-answering assistant that MUST
ground every claim in the provided context chunks.

Rules:
- Answer using ONLY information in the context chunks below.
- Cite every claim inline using the chunk's bracketed number, e.g. [1], [2].
- If multiple chunks support a claim, cite all of them, e.g. [1][3].
- Do NOT state anything not directly supported by the context — do not invent
  facts, names, or events absent from the chunks."""

def format_context(chunks):
    return "\n\n".join(
        f"[{i+1}] (source: {c['source_doc']})\n{c['text']}"
        for i, c in enumerate(chunks)
    )

def generate_answer(model, query, context_chunks, max_new_tokens=250):
    context_str = format_context(context_chunks)
    messages = [
        {"role": "system", "content": CITATION_SYSTEM_PROMPT},
        {"role": "user", "content": f"Context:\n{context_str}\n\nQuestion: {query}"},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True,
    ).to(model.device)

    t0 = time.time()
    output = model.generate(
        **inputs, max_new_tokens=max_new_tokens, temperature=0.2, do_sample=True,
    )
    latency = time.time() - t0

    text = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return text, latency

In [5]:
#run BASE model over all eval queries, checkpointed
#BASE_RESULTS_PATH = CHECKPOINT_DIR / "eval_base_results.json"
BASE_RESULTS_PATH = CHECKPOINT_DIR / "eval_qwen_base_results.json"

def load_results(path):
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))
    return {}

def save_results(path, results):
    path.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")

base_results = load_results(BASE_RESULTS_PATH)

for i, item in enumerate(eval_context, 1):
    q = item["query"]
    if q in base_results:
        print(f"[{i}/{len(eval_context)}] SKIP (base): {q[:50]!r}")
        continue
    print(f"[{i}/{len(eval_context)}] (base) {q[:50]!r}")
    answer, latency = generate_answer(base_model, q, item["context_chunks"])
    base_results[q] = {"answer": answer, "latency_seconds": round(latency, 2)}
    save_results(BASE_RESULTS_PATH, base_results)

print(f"\nBase model: {len(base_results)}/{len(eval_context)} done")

[1/15] (base) 'What is the definition of a sentence according to '
[2/15] (base) 'Who formed the Afrikander Ministry in October 1897'
[3/15] (base) 'How did Mr. Cruncher recruit his finances?'
[4/15] (base) 'How did GPUs change parallel computing according t'
[5/15] (base) "What strange fancy grew in the narrator's mind abo"
[6/15] (base) 'Why is C still popular despite newer programming l'
[7/15] (base) 'How does the StringTokenizer work in the Java code'
[8/15] (base) 'عمران نے رات کو کیا کرنے کا مشورہ دیا؟'
[9/15] (base) 'پاکستان میں پرندوں کی کتنی اقسام پائی جاتی ہیں؟'
[10/15] (base) 'تقسیم ہند کے بعد پاکستان کی قومی زبان کیا بنی؟'
[11/15] (base) 'نوآبادیاتی ہندوستانی اسلامی اسکولوں میں مسلمانوں ک'
[12/15] (base) '대한민국의 초·중·고등학교는 2012학년도부터 주5일제를 전면 실시하였다'
[13/15] (base) '미국 보수 기독교계에서는 기독교 근본주의를 주장하였다'
[14/15] (base) "괴테는 괴로움에 대해 '괴로움이 남기고 간 것을 맛보아라'고 말했다"
[15/15] (base) '5월은 그레고리력에서 한 해의 다섯 번째 달이며 31일까지 있는 7개의 달 중 하나이다'

Base model: 15/15 done


In [6]:
# attach the fine-tuned adapter on top of the SAME base model
# don't reload the base weights
from peft import PeftModel

finetuned_model = PeftModel.from_pretrained(base_model, str(ADAPTER_PATH))
finetuned_model.eval()
print("Fine-tuned adapter attached")

Fine-tuned adapter attached


In [7]:
# run FINE-TUNED model over all eval queries, checkpointed
#FT_RESULTS_PATH = CHECKPOINT_DIR / "eval_finetuned_results.json"
FT_RESULTS_PATH = CHECKPOINT_DIR / "eval_qwen_finetuned_results.json"

ft_results = load_results(FT_RESULTS_PATH)

for i, item in enumerate(eval_context, 1):
    q = item["query"]
    if q in ft_results:
        print(f"[{i}/{len(eval_context)}] SKIP (finetuned): {q[:50]!r}")
        continue
    print(f"[{i}/{len(eval_context)}] (finetuned) {q[:50]!r}")
    answer, latency = generate_answer(finetuned_model, q, item["context_chunks"])
    ft_results[q] = {"answer": answer, "latency_seconds": round(latency, 2)}
    save_results(FT_RESULTS_PATH, ft_results)

print(f"\nFine-tuned model: {len(ft_results)}/{len(eval_context)} done")

[1/15] (finetuned) 'What is the definition of a sentence according to '
[2/15] (finetuned) 'Who formed the Afrikander Ministry in October 1897'
[3/15] (finetuned) 'How did Mr. Cruncher recruit his finances?'
[4/15] (finetuned) 'How did GPUs change parallel computing according t'
[5/15] (finetuned) "What strange fancy grew in the narrator's mind abo"
[6/15] (finetuned) 'Why is C still popular despite newer programming l'
[7/15] (finetuned) 'How does the StringTokenizer work in the Java code'
[8/15] (finetuned) 'عمران نے رات کو کیا کرنے کا مشورہ دیا؟'
[9/15] (finetuned) 'پاکستان میں پرندوں کی کتنی اقسام پائی جاتی ہیں؟'
[10/15] (finetuned) 'تقسیم ہند کے بعد پاکستان کی قومی زبان کیا بنی؟'
[11/15] (finetuned) 'نوآبادیاتی ہندوستانی اسلامی اسکولوں میں مسلمانوں ک'
[12/15] (finetuned) '대한민국의 초·중·고등학교는 2012학년도부터 주5일제를 전면 실시하였다'
[13/15] (finetuned) '미국 보수 기독교계에서는 기독교 근본주의를 주장하였다'
[14/15] (finetuned) "괴테는 괴로움에 대해 '괴로움이 남기고 간 것을 맛보아라'고 말했다"
[15/15] (finetuned) '5월은 그레고리력에서 한 해의 다섯 번째 달이며 31일까지 있는 7

In [8]:
# automatic (non-judge) metrics: citation correctness, hallucination
# proxy (invented citation numbers), response length — all computable
# directly from the text, no model call needed

def extract_cited_indices(answer):
    return set(int(m) for m in re.findall(r'\[(\d+)\]', answer))

def score_answer(query, answer, context_chunks, ground_truth_chunk_id, ground_truth_retrieved):
    cited_indices = extract_cited_indices(answer)
    n_context = len(context_chunks)

    # invented citation: a number outside the range of chunks actually provided
    invented_citations = [i for i in cited_indices if i < 1 or i > n_context]

    # did the answer cite the chunk that actually contains the ground truth?
    cited_chunk_ids = {context_chunks[i-1]["chunk_id"] for i in cited_indices if 1 <= i <= n_context}
    cited_ground_truth = ground_truth_chunk_id in cited_chunk_ids

    return {
        "has_citations": len(cited_indices) > 0,
        "num_citations": len(cited_indices),
        "invented_citation_count": len(invented_citations),
        "cited_ground_truth_chunk": cited_ground_truth,
        # only a fair "did generation succeed" check if retrieval actually
        # found the right chunk in the first place
        "citation_correct_given_retrieval": cited_ground_truth if ground_truth_retrieved else None,
        "answer_length_chars": len(answer),
    }

def score_all(results_dict, eval_context):
    ctx_by_query = {item["query"]: item for item in eval_context}
    scored = {}
    for q, r in results_dict.items():
        item = ctx_by_query[q]
        scored[q] = {
            **score_answer(q, r["answer"], item["context_chunks"], item["ground_truth_chunk_id"], item["ground_truth_retrieved"]),
            "latency_seconds": r["latency_seconds"],
        }
    return scored

base_scores = score_all(base_results, eval_context)
ft_scores = score_all(ft_results, eval_context)

In [11]:
# LLM-as-judge using a genuinely independent model (openai/gpt-oss-120b
# via Groq), not the model being evaluated — avoids the self-judging bias of
# using the base model to grade both itself and the fine-tuned version
import time
from openai import OpenAI, RateLimitError, APITimeoutError
from google.colab import userdata

groq_client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=userdata.get('GROQ_API_KEY'),  # add this as a Colab secret first
)

JUDGE_MODEL = "openai/gpt-oss-120b"

JUDGE_PROMPT = """You are a strict fact-checker. Given a question, an answer,
and the source context it was supposedly based on, evaluate:

GROUNDED: Does every claim in the answer appear in or follow directly from
the context? Answer yes or no.
ADDRESSES_QUERY: Does the answer respond to what was actually asked?
Answer yes or no.

Output in this exact format, nothing else:
GROUNDED: yes/no
ADDRESSES_QUERY: yes/no"""

def call_judge(messages, max_retries=3):
    last_error = None
    for attempt in range(max_retries):
        try:
            response = groq_client.chat.completions.create(
                model=JUDGE_MODEL,
                messages=messages,
                temperature=0.0,
                max_tokens=200,
                reasoning_effort="low",
                timeout=20,
            )
            text = response.choices[0].message.content
            if text and len(text.strip()) >= 3:
                return text.strip()
            last_error = f"empty response (finish_reason={response.choices[0].finish_reason!r})"
        except RateLimitError as e:
            last_error = e
            wait = 8 * (attempt + 1)
            print(f"  rate limited, retrying in {wait}s...")
            time.sleep(wait)
        except APITimeoutError as e:
            last_error = e
            time.sleep(3)
    raise RuntimeError(f"Judge call failed after {max_retries} retries: {last_error}")

def judge_answer(query, answer, context_chunks):
    context_str = format_context(context_chunks)
    messages = [
        {"role": "system", "content": JUDGE_PROMPT},
        {"role": "user", "content": f"Question: {query}\n\nAnswer: {answer}\n\nContext:\n{context_str}"},
    ]
    response = call_judge(messages)
    grounded_match = re.search(r'GROUNDED\s*:\s*(\w+)', response, re.IGNORECASE)
    addresses_match = re.search(r'ADDRESSES_QUERY\s*:\s*(\w+)', response, re.IGNORECASE)
    return {
        "grounded": bool(grounded_match and grounded_match.group(1).lower().startswith('y')),
        "addresses_query": bool(addresses_match and addresses_match.group(1).lower().startswith('y')),
    }

#JUDGE_RESULTS_PATH = CHECKPOINT_DIR / "eval_judge_results.json"
JUDGE_RESULTS_PATH = CHECKPOINT_DIR / "eval_qwen_judge_results.json"
judge_results = load_results(JUDGE_RESULTS_PATH)

for label, results_dict in [("base", base_results), ("finetuned", ft_results)]:
    for i, item in enumerate(eval_context, 1):
        q = item["query"]
        key = f"{label}::{q}"
        if key in judge_results:
            print(f"[{label} {i}/{len(eval_context)}] SKIP: {q[:50]!r}")
            continue
        print(f"[{label} {i}/{len(eval_context)}] judging: {q[:50]!r}")
        verdict = judge_answer(q, results_dict[q]["answer"], item["context_chunks"])
        judge_results[key] = verdict
        save_results(JUDGE_RESULTS_PATH, judge_results)

print("Judging complete")

[base 1/15] SKIP: 'What is the definition of a sentence according to '
[base 2/15] SKIP: 'Who formed the Afrikander Ministry in October 1897'
[base 3/15] SKIP: 'How did Mr. Cruncher recruit his finances?'
[base 4/15] SKIP: 'How did GPUs change parallel computing according t'
[base 5/15] SKIP: "What strange fancy grew in the narrator's mind abo"
[base 6/15] SKIP: 'Why is C still popular despite newer programming l'
[base 7/15] SKIP: 'How does the StringTokenizer work in the Java code'
[base 8/15] SKIP: 'عمران نے رات کو کیا کرنے کا مشورہ دیا؟'
[base 9/15] SKIP: 'پاکستان میں پرندوں کی کتنی اقسام پائی جاتی ہیں؟'
[base 10/15] SKIP: 'تقسیم ہند کے بعد پاکستان کی قومی زبان کیا بنی؟'
[base 11/15] SKIP: 'نوآبادیاتی ہندوستانی اسلامی اسکولوں میں مسلمانوں ک'
[base 12/15] SKIP: '대한민국의 초·중·고등학교는 2012학년도부터 주5일제를 전면 실시하였다'
[base 13/15] SKIP: '미국 보수 기독교계에서는 기독교 근본주의를 주장하였다'
[base 14/15] SKIP: "괴테는 괴로움에 대해 '괴로움이 남기고 간 것을 맛보아라'고 말했다"
[base 15/15] SKIP: '5월은 그레고리력에서 한 해의 다섯 번째 달이며 31일까지 있는 7개의 달 중 하나이다'
[f

In [12]:
#assemble final comparison summary and save for local report writing
def summarize(scores, judge_results, label):
    n = len(scores)
    hallucination_free = sum(1 for s in scores.values() if s["invented_citation_count"] == 0) / n
    has_citations_rate = sum(1 for s in scores.values() if s["has_citations"]) / n
    retrieval_ok = [s for s in scores.values() if s["citation_correct_given_retrieval"] is not None]
    citation_accuracy = (sum(1 for s in retrieval_ok if s["citation_correct_given_retrieval"]) / len(retrieval_ok)) if retrieval_ok else None
    avg_latency = sum(s["latency_seconds"] for s in scores.values()) / n

    judge_grounded = [v["grounded"] for k, v in judge_results.items() if k.startswith(f"{label}::")]
    judge_addresses = [v["addresses_query"] for k, v in judge_results.items() if k.startswith(f"{label}::")]

    return {
        "n_queries": n,
        "hallucination_free_rate": round(hallucination_free, 3),
        "has_citations_rate": round(has_citations_rate, 3),
        "citation_accuracy_given_correct_retrieval": round(citation_accuracy, 3) if citation_accuracy is not None else None,
        "n_queries_with_correct_retrieval": len(retrieval_ok),
        "judge_grounded_rate": round(sum(judge_grounded) / len(judge_grounded), 3) if judge_grounded else None,
        "judge_addresses_query_rate": round(sum(judge_addresses) / len(judge_addresses), 3) if judge_addresses else None,
        "avg_latency_seconds": round(avg_latency, 2),
    }

summary = {
    "base": summarize(base_scores, judge_results, "base"),
    "finetuned": summarize(ft_scores, judge_results, "finetuned"),
}

print(json.dumps(summary, indent=2))

FINAL_SUMMARY_PATH = CHECKPOINT_DIR / "qwen_comparison_summary.json"
FINAL_SUMMARY_PATH.write_text(json.dumps({
    "summary": summary,
    "base_scores": base_scores,
    "finetuned_scores": ft_scores,
    "judge_results": judge_results,
}, ensure_ascii=False, indent=2), encoding="utf-8")

files.download(str(FINAL_SUMMARY_PATH))
print(f"\nSaved and downloading {FINAL_SUMMARY_PATH.name}.")

{
  "base": {
    "n_queries": 15,
    "hallucination_free_rate": 0.933,
    "has_citations_rate": 1.0,
    "citation_accuracy_given_correct_retrieval": 0.727,
    "n_queries_with_correct_retrieval": 11,
    "judge_grounded_rate": 0.933,
    "judge_addresses_query_rate": 0.933,
    "avg_latency_seconds": 22.1
  },
  "finetuned": {
    "n_queries": 15,
    "hallucination_free_rate": 1.0,
    "has_citations_rate": 1.0,
    "citation_accuracy_given_correct_retrieval": 1.0,
    "n_queries_with_correct_retrieval": 11,
    "judge_grounded_rate": 0.733,
    "judge_addresses_query_rate": 0.867,
    "avg_latency_seconds": 19.7
  }
}


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Saved and downloading qwen_comparison_summary.json.


In [13]:
# Cell 9b — same judge logic, second independent model for cross-validation
JUDGE_MODEL_2 = "llama-3.3-70b-versatile"

def call_judge_v2(messages, max_retries=3):
    last_error = None
    for attempt in range(max_retries):
        try:
            response = groq_client.chat.completions.create(
                model=JUDGE_MODEL_2,
                messages=messages,
                temperature=0.0,
                max_tokens=60,   # plain instruct model, no reasoning tokens — 60 is plenty
                timeout=20,
            )
            text = response.choices[0].message.content
            if text and len(text.strip()) >= 3:
                return text.strip()
            last_error = f"empty response (finish_reason={response.choices[0].finish_reason!r})"
        except RateLimitError as e:
            last_error = e
            wait = 8 * (attempt + 1)
            print(f"  rate limited, retrying in {wait}s...")
            time.sleep(wait)
        except APITimeoutError as e:
            last_error = e
            time.sleep(3)
    raise RuntimeError(f"Judge call failed after {max_retries} retries: {last_error}")

def judge_answer_v2(query, answer, context_chunks):
    context_str = format_context(context_chunks)
    messages = [
        {"role": "system", "content": JUDGE_PROMPT},
        {"role": "user", "content": f"Question: {query}\n\nAnswer: {answer}\n\nContext:\n{context_str}"},
    ]
    response = call_judge_v2(messages)
    grounded_match = re.search(r'GROUNDED\s*:\s*(\w+)', response, re.IGNORECASE)
    addresses_match = re.search(r'ADDRESSES_QUERY\s*:\s*(\w+)', response, re.IGNORECASE)
    return {
        "grounded": bool(grounded_match and grounded_match.group(1).lower().startswith('y')),
        "addresses_query": bool(addresses_match and addresses_match.group(1).lower().startswith('y')),
    }

#JUDGE_V2_RESULTS_PATH = CHECKPOINT_DIR / "eval_judge_results_llama.json"
JUDGE_V2_RESULTS_PATH = CHECKPOINT_DIR / "eval_qwen_judge_results_llama.json"
judge_results_v2 = load_results(JUDGE_V2_RESULTS_PATH)

for label, results_dict in [("base", base_results), ("finetuned", ft_results)]:
    for i, item in enumerate(eval_context, 1):
        q = item["query"]
        key = f"{label}::{q}"
        if key in judge_results_v2:
            print(f"[{label} {i}/{len(eval_context)}] SKIP: {q[:50]!r}")
            continue
        print(f"[{label} {i}/{len(eval_context)}] judging (llama): {q[:50]!r}")
        verdict = judge_answer_v2(q, results_dict[q]["answer"], item["context_chunks"])
        judge_results_v2[key] = verdict
        save_results(JUDGE_V2_RESULTS_PATH, judge_results_v2)

print("Second-judge pass complete")

[base 1/15] judging (llama): 'What is the definition of a sentence according to '
[base 2/15] judging (llama): 'Who formed the Afrikander Ministry in October 1897'
[base 3/15] judging (llama): 'How did Mr. Cruncher recruit his finances?'
[base 4/15] judging (llama): 'How did GPUs change parallel computing according t'
[base 5/15] judging (llama): "What strange fancy grew in the narrator's mind abo"
[base 6/15] judging (llama): 'Why is C still popular despite newer programming l'
[base 7/15] judging (llama): 'How does the StringTokenizer work in the Java code'
[base 8/15] judging (llama): 'عمران نے رات کو کیا کرنے کا مشورہ دیا؟'
[base 9/15] judging (llama): 'پاکستان میں پرندوں کی کتنی اقسام پائی جاتی ہیں؟'
[base 10/15] judging (llama): 'تقسیم ہند کے بعد پاکستان کی قومی زبان کیا بنی؟'
[base 11/15] judging (llama): 'نوآبادیاتی ہندوستانی اسلامی اسکولوں میں مسلمانوں ک'
[base 12/15] judging (llama): '대한민국의 초·중·고등학교는 2012학년도부터 주5일제를 전면 실시하였다'
[base 13/15] judging (llama): '미국 보수 기독교계에서는 기독교 근

In [14]:
# inter-judge agreement
agree_grounded = sum(
    1 for k in judge_results
    if k in judge_results_v2 and judge_results[k]["grounded"] == judge_results_v2[k]["grounded"]
)
agree_addresses = sum(
    1 for k in judge_results
    if k in judge_results_v2 and judge_results[k]["addresses_query"] == judge_results_v2[k]["addresses_query"]
)
n = len(judge_results)

print(f"Judge agreement — grounded: {agree_grounded}/{n} ({agree_grounded/n:.1%})")
print(f"Judge agreement — addresses_query: {agree_addresses}/{n} ({agree_addresses/n:.1%})")

disagreements = [
    k for k in judge_results
    if k in judge_results_v2 and judge_results[k]["grounded"] != judge_results_v2[k]["grounded"]
]
print(f"\nDisagreement cases (grounded): {disagreements}")

Judge agreement — grounded: 27/30 (90.0%)
Judge agreement — addresses_query: 29/30 (96.7%)

Disagreement cases (grounded): ['base::Who formed the Afrikander Ministry in October 1897?', 'finetuned::How did GPUs change parallel computing according to the text?', 'finetuned::대한민국의 초·중·고등학교는 2012학년도부터 주5일제를 전면 실시하였다']


In [15]:
# Cell 9d — print the disagreement cases side by side for manual review
disagreement_keys = [
    'finetuned::How did GPUs change parallel computing according to the text?',
    'finetuned::Why is C still popular despite newer programming languages?',
    'finetuned::How does the StringTokenizer work in the Java code example?',
    'finetuned::대한민국의 초·중·고등학교는 2012학년도부터 주5일제를 전면 실시하였다',
    'base::عمران نے رات کو کیا کرنے کا مشورہ دیا؟',
    'finetuned::عمران نے رات کو کیا کرنے کا مشورہ دیا؟',
]

for key in disagreement_keys:
    label, q = key.split("::", 1)
    results_dict = base_results if label == "base" else ft_results
    print(f"\n{'='*70}\n{key}")
    print(f"gpt-oss:  grounded={judge_results[key]['grounded']}")
    print(f"llama:    grounded={judge_results_v2[key]['grounded']}")
    print(f"\nAnswer: {results_dict[q]['answer'][:400]}")


finetuned::How did GPUs change parallel computing according to the text?
gpt-oss:  grounded=True
llama:    grounded=False

Answer: GPUs changed parallel computing by making it feasible for mass-market products, thus increasing its market presence and economic attractiveness for application developers [1]. GPUs are now widely used in PCs, with over 200 million units of the G80 processors and their successors sold to date [2]. This large market presence has made massively parallel computing accessible to a broader range of appl

finetuned::Why is C still popular despite newer programming languages?
gpt-oss:  grounded=True
llama:    grounded=True

Answer: C remains popular despite newer programming languages due to its reliability, simplicity, and performance [1]. It is used extensively in operating systems and device drivers where speed and efficiency are critical [1][2]. Additionally, many modern languages are built on C, maintaining its relevance [1].

finetuned::How does the StringTo

In [16]:
def has_repetition(answer, min_repeat_len=40):
    """Flags near-exact repeated substrings — a common degenerate-generation
    signal that citation/hallucination checks don't catch."""
    for i in range(len(answer) - min_repeat_len):
        chunk = answer[i:i+min_repeat_len]
        if answer.count(chunk) > 1:
            return True
    return False

for label, results in [("base", base_results), ("finetuned", ft_results)]:
    flagged = [q for q, r in results.items() if has_repetition(r["answer"])]
    print(f"{label}: {len(flagged)} answers with repeated text")
    for q in flagged:
        print(f"  - {q[:60]}")

base: 1 answers with repeated text
  - Who formed the Afrikander Ministry in October 1897?
finetuned: 0 answers with repeated text


In [ ]:
# consolidate everything into one final file: original comparison
# summary + llama judge results + agreement stats + repetition check
import json
from pathlib import Path

CHECKPOINT_DIR = Path("/content/drive/MyDrive/finetuning_checkpoints")

def load_json(path):
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}

base_results = load_json(CHECKPOINT_DIR / "eval_base_results.json")
ft_results = load_json(CHECKPOINT_DIR / "eval_finetuned_results.json")
judge_results_gptoss = load_json(CHECKPOINT_DIR / "eval_judge_results.json")
judge_results_llama = load_json(CHECKPOINT_DIR / "eval_judge_results_llama.json")

# recompute agreement stats (cheap, no API calls)
n = len(judge_results_gptoss)
agree_grounded = sum(
    1 for k in judge_results_gptoss
    if k in judge_results_llama and judge_results_gptoss[k]["grounded"] == judge_results_llama[k]["grounded"]
)
agree_addresses = sum(
    1 for k in judge_results_gptoss
    if k in judge_results_llama and judge_results_gptoss[k]["addresses_query"] == judge_results_llama[k]["addresses_query"]
)
disagreement_keys_grounded = [
    k for k in judge_results_gptoss
    if k in judge_results_llama and judge_results_gptoss[k]["grounded"] != judge_results_llama[k]["grounded"]
]

# repetition check (same function as before)
def has_repetition(answer, min_repeat_len=40):
    for i in range(len(answer) - min_repeat_len):
        chunk = answer[i:i+min_repeat_len]
        if answer.count(chunk) > 1:
            return True
    return False

repetition_flags = {
    "base": [q for q, r in base_results.items() if has_repetition(r["answer"])],
    "finetuned": [q for q, r in ft_results.items() if has_repetition(r["answer"])],
}

final = {
    "base_results": base_results,
    "finetuned_results": ft_results,
    "judge_results_gptoss": judge_results_gptoss,
    "judge_results_llama": judge_results_llama,
    "judge_agreement": {
        "grounded_agreement_rate": round(agree_grounded / n, 3) if n else None,
        "addresses_query_agreement_rate": round(agree_addresses / n, 3) if n else None,
        "grounded_disagreement_cases": disagreement_keys_grounded,
    },
    "repetition_flags": repetition_flags,
}

FINAL_PATH = CHECKPOINT_DIR / "final_comparison_summary.json"
FINAL_PATH.write_text(json.dumps(final, ensure_ascii=False, indent=2), encoding="utf-8")

from google.colab import files
files.download(str(FINAL_PATH))
print(f"Consolidated and downloaded: {FINAL_PATH.name}")

## cross model comparison

In [17]:
# Cell — build a single file with all four result sets, for both
# same-model and cross-model comparison in the dashboard
import json
from pathlib import Path

LLAMA_DIR = Path("/content/drive/MyDrive/finetuning_checkpoints")
QWEN_DIR = Path("/content/drive/MyDrive/finetuning_checkpoints_qwen")

def load(path):
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}

combined = {
    "llama_base": load(LLAMA_DIR / "eval_base_results.json"),
    "llama_finetuned": load(LLAMA_DIR / "eval_finetuned_results.json"),
    "llama_judge": load(LLAMA_DIR / "eval_judge_results.json"),
    "llama_judge_v2": load(LLAMA_DIR / "eval_judge_results_llama.json"),
    "llama_repetition_flags": load(LLAMA_DIR / "eval_judge_results.json").get("repetition_flags", {}),

    "qwen_base": load(QWEN_DIR / "eval_qwen_base_results.json"),
    "qwen_finetuned": load(QWEN_DIR / "eval_qwen_finetuned_results.json"),
    "qwen_judge": load(QWEN_DIR / "eval_qwen_judge_results.json"),
}

OUT_PATH = Path("/content/drive/MyDrive/finetuning_checkpoints/cross_model_comparison.json")
OUT_PATH.write_text(json.dumps(combined, ensure_ascii=False, indent=2), encoding="utf-8")

from google.colab import files
files.download(str(OUT_PATH))
print(f"Saved and downloaded: {OUT_PATH.name}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Saved and downloaded: cross_model_comparison.json
